In [ ]:
import os
import bs4
import time
import pinecone
import warnings
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain import hub
from langchain.schema import Document
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_core.retrievers import BaseRetriever
from langchain_openai import ChatOpenAI
from langchain_core.prompt_values import ChatPromptValue
from langchain_core.runnables import RunnableLambda
import scipy.spatial.distance as spdist
from typing import List
from typing import Dict
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")

In [ ]:
# get API keys 
load_dotenv("/Users/kattygeng/Desktop/MLDS/424-genAI/genai_key.env")  # change to your file path to obtain api keys 
openai_api_key = os.environ.get("OPENAI_API_KEY")
pinecone_api_key = os.environ.get("PINECONE_API_KEY")

In [129]:
class EmbeddingModel:
    """
    A class to handle embedding of documents and queries using a sentence transformer model.
    """

    def __init__(self, model):
        """
        Initialize the EmbeddingModel class with a specified SentenceTransformer model.
        """
        self.model = SentenceTransformer(model)

    def embed_documents(self, docs):
        """
        Embed a list of documents into vector representations.
        """
        return self.model.encode(docs).tolist()

    def embed_query(self, doc):
        """
        Embed a single query document into a vector representation.
        """
        return self.model.encode(doc).tolist()

In [130]:
# initiate pinecone and vectorstore
pinecone_client = pinecone.Pinecone(api_key = pinecone_api_key, environment="us-east-1")
index_name = "recipe-index"
model_name = 'DivyaMereddy007/RecipeBert_v5'
embeddings = EmbeddingModel(model = model_name)

vectorstore = PineconeVectorStore(
    index_name=index_name,
    embedding=embeddings,
    pinecone_api_key=pinecone_api_key
)

In [131]:
class CustomRetriever(BaseRetriever):
    ''' 
    A custom retriever that performs similarity search using PineconeVectorStore and returns the raw text content of the most relevant documents
    '''
    vectorstore: PineconeVectorStore

    def _get_relevant_documents(self, query: str):
        '''
        Perform a similarity search on the vector store and retrieve the raw text of the top 10 most relevant documents.
        '''
        docs = self.vectorstore.similarity_search(query, k=5)
        outputs = []
        for doc in docs:
            raw_text = doc.page_content # Access the raw text from metadata
            outputs.append(raw_text)
        return outputs
retriever = CustomRetriever(vectorstore=vectorstore)

In [132]:
def format_docs(docs):
    '''
    Formats a list of documents into a single string, separating each document with two newline characters.
    '''
    return "\n\n".join(docs)

### LLM

In [133]:
prompt='''Provide me with 3 easy-to-make Chinese cuisine for dinner.'''

In [134]:
template_prompt = hub.pull("rlm/rag-prompt")
llm = ChatOpenAI(model="gpt-3.5-turbo", seed=0)

In [135]:
llm_chain = (
    llm
    | StrOutputParser()
)

In [ ]:
# example output 
llm_chain.invoke(prompt)

'1. Stir-fried noodles with vegetables: Cook noodles according to package instructions. In a hot pan, stir-fry your favorite vegetables with garlic, ginger, soy sauce, and a splash of sesame oil. Add cooked noodles and toss until well combined. Serve hot with a sprinkle of green onions on top.\n\n2. Sweet and sour tofu: Cut tofu into cubes and toss in cornstarch. Fry until crispy and set aside. In a separate pan, sauté bell peppers, onions, and pineapple chunks in a sweet and sour sauce made from ketchup, vinegar, sugar, and soy sauce. Add fried tofu back into the pan and stir until coated. Serve over rice.\n\n3. Kung Pao chicken: Marinate chicken pieces in soy sauce, rice wine, and cornstarch. Stir-fry chicken in a hot pan until cooked through. Add peanuts, dried red chilies, and diced bell peppers. In a separate bowl, mix soy sauce, vinegar, sugar, and cornstarch to make the sauce. Pour sauce over the chicken and stir until everything is coated. Serve hot with steamed rice.'

### RAG

In [137]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | template_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# example output 
rag_chain.invoke(prompt)

'Three easy-to-make Chinese cuisine for dinner are Chinese Style Marinade for meat, Quick Pickled Radish Salad With Garlic as a side dish, and Stir-Fried Chinese Vegetables as a vegetable dish.'

### RAG with advanced functions

In [ ]:
def label_with_llm(llm, query, retrieved_doc):
    ''' 
    Classifies and labels recipes with cuisine type and estimated cook time using an LLM.
    '''
    labeled_docs = []
    for doc in retrieved_doc.split("\n\n"):
        cuisine_prompt = f"""
        Document: {doc}
        Classify and label what type of cuisine is this recipe. Return 'cuisine type:' + only the label with no other words.
        Choose from the following list: Italian, French, Chinese, Japanese, Indian, Thai, Mexican, Spanish, Greek, Middle Eastern, 
        Korean, Vietnamese, American, British, German, Brazilian, Moroccan, Turkish, Ethiopian, Peruvian, Caribbean, Mediterranean, 
        Cajun, Polynesian, Scandinavian, Russian, Filipino, Malaysian, Indonesian, South African, and Asian Fusion.
        """
        cuisine = llm.invoke(cuisine_prompt)  

        time_prompt = f"""
        Document: {doc}
        Estimate the amount of time needed to cook this recipe. Return 'cook time:' + only the cook time as a number with no other words. 
        """
        time = llm.invoke(time_prompt)
        labeled_docs.append((doc, cuisine.content, time.content))
    
    return labeled_docs

In [ ]:
def rewrite_query_hyde(input_dict: Dict[str, str]) -> str:
    """
    Implements query rewriting using HyDE (Hypothetical Document Embeddings). Given an input dictionary containing a query, 
    this function generates a hypothetical document using an LLM and then refines the query for better retrieval performance.
    """
    original_query = input_dict["original_prompt"]

    hyde_prompt = f'''
        Imagine an ideal response to the following query, as if it were extracted from a high-quality document: "{original_query}".
        Write a relevant passage that could appear in a document answering this question.
    '''
    hypothetical_doc = llm.invoke(hyde_prompt)
    
    rewrite_prompt = f'''
        Given the original query: "{original_query}"
        and this hypothetical document: "{hypothetical_doc}"
        Rewrite the query to improve retrieval quality for my RAG model.
    '''
    rewritten_prompt = llm.invoke(rewrite_prompt)
    return rewritten_prompt 

In [ ]:
def advanced_rag(prompt):
    ''' 
    Form the advanced RAG pipeline by incorporating document labeling and query rewriting functions above.
    '''
    rag_chain_retriever = (retriever | format_docs)
    retrieved_doc = rag_chain_retriever.invoke(prompt)
    labeled_docs = label_with_llm(llm, prompt, retrieved_doc)
    #print(labeled_docs)
    rewritten_prompt = rewrite_query_hyde({"retrieved_doc": labeled_docs, "original_prompt": prompt}) 
    #print(rewritten_prompt) 
    return rag_chain.invoke(rewritten_prompt.content)

In [ ]:
# example output 
advanced_rag(prompt)

'Three simple Chinese cuisine recipes suitable for a dinner meal are Mandarin Chow Mein, Chinese Style Marinade, and Simple Chinese Noodles.'

### Test with different questions

In [ ]:
# Dataset-specific retrieval
prompt_1 = "How to make Chocolate Frango Mints?"
print("LLM: ", llm_chain.invoke(prompt_1))
print("RAG: ", rag_chain.invoke(prompt_1))
print("Advanced RAG: ", advanced_rag(prompt_1))

LLM:  Ingredients:
- 1 cup chocolate chips
- 1/4 cup heavy cream
- 1/2 teaspoon peppermint extract
- 1/2 cup powdered sugar
- Green food coloring (optional)
- 1/4 teaspoon vanilla extract

Instructions:
1. In a small saucepan, melt the chocolate chips and heavy cream over low heat, stirring constantly until smooth.
2. Remove the saucepan from the heat and stir in the peppermint extract and vanilla extract. If desired, add a few drops of green food coloring for a minty green color.
3. Gradually stir in the powdered sugar until well combined and smooth.
4. Pour the mixture into a shallow dish lined with parchment paper and refrigerate for about 1 hour, or until firm.
5. Once the mixture is firm, use a small cookie cutter or knife to cut out small squares or rectangles.
6. Store the chocolate frango mints in an airtight container in the refrigerator until ready to serve.

Enjoy your delicious homemade chocolate frango mints!
RAG:  To make Chocolate Frango Mints, mix confectioners sugar, b

Chocolate Frango Mints is one of the recipes in the dataset. Both RAG and advanced RAG is able to retrieve dataset-specific information while LLM is just generating information based on the prompt. 

In [161]:
# Inexistent recipe identification
prompt_2 = "Do you have a recipe for Sichuan peppercorn and soy sauce infused chocolate mousse?"
print("LLM: ", llm_chain.invoke(prompt_2))
print("RAG: ", rag_chain.invoke(prompt_2))
print("Advanced RAG: ", advanced_rag(prompt_2))

LLM:  Here is a recipe for Sichuan peppercorn and soy sauce infused chocolate mousse:

Ingredients:
- 200g dark chocolate (70% cocoa)
- 1 cup heavy cream
- 2 tbsp Sichuan peppercorns
- 2 tbsp soy sauce
- 3 egg yolks
- 2 tbsp sugar
- Pinch of salt
- Sea salt flakes, for garnish

Instructions:
1. In a small saucepan, heat the heavy cream over medium heat until it just begins to simmer. Add the Sichuan peppercorns and soy sauce, then remove from heat and let steep for 10 minutes.
2. Strain the cream mixture to remove the peppercorns, then return the mixture to the saucepan and heat again until simmering.
3. In a separate bowl, whisk together the egg yolks, sugar, and a pinch of salt. Slowly pour the hot cream mixture into the egg yolk mixture, whisking constantly.
4. Return the mixture to the saucepan and cook over low heat, stirring constantly, until it thickens slightly and coats the back of a spoon.
5. Remove from heat and add the chopped dark chocolate, stirring until melted and smoot

Sichuan peppercorn and soy sauce infused chocolate mousse is something I made up and it does not make sense as normal food. However, LLM is still giving me the instructions to make it based on the name itself. RAG-based models clearly do not have information about this recipe, so they answered with either 'idk' or 'do not have the recipe'. 

In [154]:
# Creativity and synthesis 
prompt_3 = "Based on the recipes in the dataset, suggest a full three-course meal that maintains a cohesive flavor profile. Provide reasonings for the choices."
print("LLM: ", llm_chain.invoke(prompt_3))
print("RAG: ", rag_chain.invoke(prompt_3))
print("Advanced RAG: ", advanced_rag(prompt_3))

LLM:  Starter: Caprese Salad

- The Caprese Salad with fresh mozzarella, ripe tomatoes, and fragrant basil is a classic Italian dish that sets a light and refreshing tone for the meal. The combination of creamy cheese, juicy tomatoes, and aromatic herbs provides a well-balanced and flavorful start to the meal.

Main Course: Chicken Alfredo Pasta

- Chicken Alfredo Pasta is a rich and indulgent dish that complements the lightness of the Caprese Salad. The creamy Alfredo sauce coats the tender chicken and pasta, creating a decadent and satisfying main course. The flavors of the Parmesan cheese and garlic in the Alfredo sauce tie in well with the basil in the Caprese Salad, creating a cohesive flavor profile for the meal.

Dessert: Tiramisu

- Tiramisu is a classic Italian dessert that rounds out the meal with its delicate layers of coffee-soaked ladyfingers and creamy mascarpone mixture. The hint of coffee flavor in the dessert complements the savory notes of the Chicken Alfredo Pasta, w

LLM answers this question with no information from the dataset. RAG does answer with specific recipes from the dataset but 'Memes Meal' does not make sense by itself. Advanced RAG not only answers with information from the dataset but also has good choices for the three courses.

In [ ]:
# Cuisine classification and reasoning
prompt_4 = "Provide me with a spicy Asian cuisine for dinner."
print("LLM: ", llm_chain.invoke(prompt_4))
print("RAG: ", rag_chain.invoke(prompt_4))
print("Advanced RAG: ", advanced_rag(prompt_4))

LLM:  Spicy Szechuan Beef with stir-fried vegetables and steamed rice.
RAG:  Cajun Sea Scallops would be a spicy Asian cuisine option for dinner. The dish involves sauteing scallops with Cajun seasoning, hot sauce, and garlic in a skillet. It offers a flavorful and spicy seafood option for a meal.
Advanced RAG:  An option similar to a Spicy Szechuan Stir-Fry dish would be a Hot And Spicy Asian Marinade, which includes ingredients like soy sauce, sesame oil, garlic, and ginger. Another choice could be Spicy Asian Dumpling Sauce with soy sauce, lemon juice, garlic, and green chili peppers for a kick of spice. The Asian Dipping Sauce with chili paste, sesame oil, and wasabi is also a flavorful option for a spicy Asian dinner.


I implemented the cuisine classification function in advanced RAG. For this question, I ask for spicy asian cuisine, but RAG gives a cajun recipe. Advanced does it better by giving authentic asian recipes. LLM still performs well by generating answer with its trained knowledge but not relating to this dataset. 

In [166]:
# Dataset-specific variations
prompt_5 = "How does the chocolate cake recipe in your dataset differ from a traditional one? State the names of the recipes identified."
print("LLM: ", llm_chain.invoke(prompt_5))
print("RAG: ", rag_chain.invoke(prompt_5))
print("Advanced RAG: ", advanced_rag(prompt_5))

LLM:  The chocolate cake recipe in my dataset is called "Decadent Chocolate Cake" and it differs from a traditional chocolate cake recipe in a few ways:

1. The recipe includes coffee as one of the ingredients, which can enhance the chocolate flavor of the cake.
2. The recipe also calls for buttermilk, which can add moisture and a slight tanginess to the cake.
3. The frosting for the cake is a rich chocolate ganache, which is a bit more decadent than a traditional buttercream frosting.

Let me know if you would like me to provide the full recipe for the "Decadent Chocolate Cake" in my dataset.
RAG:  The Spago'S Chocolate Chiffon Cake recipe differs from a traditional one by using oil and separating egg whites for folding into the batter. The Torta Alla Gianduia (Chocolate Hazelnut Cake) recipe deviates by incorporating Nutella and ground hazelnuts into the batter. The Craig Claiborne’S Rum Chocolate Dessert recipe stands out with the addition of rum and heavy cream for a rich texture.


In this example, LLM is "lying" by stating how the recipe from its dataset differs from a traditional one. RAG and advanced RAG point out names of the recipes in the dataset and states the differences. 